# Ingestion Pipeline

In [2]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [4]:
from langchain_core.documents import Document

In [6]:
#Data=>Documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [7]:
def load_all_pdfs():
  folder_path="data/pdfs"
  num_docs=0
  all_docs=[]
  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      #complete file path

      pdf_path = os.path.join(folder_path, filename)

      loader = PyPDFLoader(pdf_path)
      doc = loader.load()

      all_docs.extend(doc)
      num_docs += 1

  print("Total pdfs: ",num_docs)
  print("Total pages: ",len(all_docs))
  return all_docs

In [8]:
all_pdf_documents = load_all_pdfs()

Total pdfs:  2
Total pages:  32


### Chunks

In [10]:
# !pip install langchain_text_splitters

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [12]:
chunks = split_docs(all_pdf_documents)

In [13]:
len(chunks)

244

### Embeddings

In [14]:
from sentence_transformers import SentenceTransformer

In [15]:
class EmbeddingManager:
  def __init__(self, model_name="all-MiniLM-L6-v2"):

    self.model_name = model_name
    print("loading model....", self.model_name)
    self.model = SentenceTransformer(self.model_name)
    print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

  def generate_embeddings(self, text):
    embeddings = self.model.encode(text, show_progress_bar=True)
    print("embedding shape=",embeddings.shape)
    return embeddings

In [16]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions= 384


/tmp/ipykernel_1750/1861970176.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


### Vector Store

In [17]:
import chromadb
import uuid

In [18]:
class VectorStoreManager:
  def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.collection = None
    self.client = None

    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_directory, exist_ok=True)
    #create a client
    self.client = chromadb.PersistentClient(path=self.persist_directory)

    #create the collection
    self.collection = self.client.get_or_create_collection(
        name = self.collection_name,
        metadata = {"description": "vector store collection for pdf embeddings in RAG"}
    )

    print("initialized the vector store with collection=", self.collection_name)
    print("docs in collection=", self.collection.count())

  def add_documents(self, documents, embeddings):
    if len(documents) != len(embeddings):
      raise ValueError("num of chunked documents does not match num of embeddings")

    # store => ids,embedding,page-content,metadata
    ids = []
    all_metadata = []
    documents_content = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata["doc_index"] = i
      metadata["content_length"] = len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)

      embeddings_list.append(embedding.tolist())

    self.collection.add(
        ids=ids,
        metadatas=all_metadata,
        documents=documents_content,
        embeddings=embeddings_list
    )

    print("total documents added in vector store=", len(documents_content))
    print("docs in collection=", self.collection.count())

In [19]:
vector_store = VectorStoreManager()

initialized the vector store with collection= pdf_documents
docs in collection= 0


In [20]:
# chunks => embeddings

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embedding shape= (244, 384)
total documents added in vector store= 244
docs in collection= 244


# Retrieval Pipeline

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
class RAGRetriever:
  def __init__(self, embedding_manager, vector_store):
    self.embedding_manager = embedding_manager
    self.vector_store = vector_store

  def retrieve(self, query, top_k=5, score_threshold=0):
    # query=>embedding
    query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

    # semantic search
    results = self.vector_store.collection.query(
        query_embeddings = [query_embeddings.tolist()],
        n_results = top_k
    )

    # cosine similarity
    retrieved_docs = []
    if results["documents"] and results["documents"][0]:
      ids = results["ids"][0]
      metadatas = results["metadatas"][0]
      documents = results["documents"][0]
      distances = results["distances"][0]

      for i, (id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
        similarity_score = 1 - distance

        if similarity_score >= score_threshold:
          retrieved_docs.append({
              "id":id,
              "document":document,
              "metadata":metadata,
              "distance":distance,
              "similarity_score":similarity_score,
              "rank": i + 1
          })

        print(f"Retrieved {len(retrieved_docs)} documents")

    else:
      print("no documents found")

    return retrieved_docs

In [23]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [25]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape= (1, 384)
Retrieved 1 documents
Retrieved 2 documents
Retrieved 3 documents
Retrieved 4 documents
Retrieved 5 documents


[{'id': 'doc_133f0c40-8a0d-4a57-be5c-f8eeaa7e40cc',
  'document': 'Our contributions are as follows:\n\ue000 In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'producer': 'cairo 1.18.0 (https://cairographics.org)',
   'page_label': '1',
   'creator': 'Mozilla Firefox 147.0.3',
   'creationdate': '2026-06-09T14:28:50+05:30',
   'content_length': 238,
   'doc_index': 11,
   'total_pages': 21,
   'source': 'data/pdfs/research2.pdf',
   'page': 0},
  'distance': 0.4401114284992218,
  'similarity_score': 0.5598885715007782,
  'rank': 1},
 {'id': 'doc_7f76c588-4a38-4456-87f1-0cc8ce5817a8',
  'document': 'effective RAG framework.\n\ue000 We have summarized the current assessment methods of\nRAG, covering 26 tasks, nearly 50 datasets, outlining\nthe evaluation objectives and metrics, as well as the\ncurrent evaluation bench

# Integrate with LLMs

## OpenAI - GPT

In [28]:
# !pip install langchain-OpenAI

In [29]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.5",
    temperature=0.1,
    max_tokens=1024
)

In [32]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k=3):
  results = retriever.retrieve(query, top_k)

  context = "\n".join([doc["document"] for doc in results]) if results else ""

  if not context:
    print("no relevant context found")

  # prompt => context + query
  prompt = f""" use given context to generate the answer for the query
              Context: {context}
              Query: {query}"""

  response = llm.invoke(prompt) # GPT LLM expects a string as prompt
  return response.content

In [35]:
# answer = generate_output("What is RAG?", rag_retriever, llm)

## Groq

In [38]:
# !pip install langchain-groq

In [45]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_KEY_Groq,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [46]:
# generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k=3):
  results = retriever.retrieve(query, top_k)

  context = "\n".join([doc["document"] for doc in results]) if results else ""

  if not context:
    print("no relevant context found")

  # prompt => context + query
  prompt = f""" use given context to generate the answer for the query
              Context: {context}
              Query: {query}"""

  response = llm.invoke([prompt.format(context=context, query=query)]) # Groq LLM expects a list as prompt
  return response.content

In [47]:
answer = generate_output("What is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape= (1, 384)
Retrieved 1 documents
Retrieved 2 documents
Retrieved 3 documents


In [49]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" I need to use the provided context to answer this. Let me read through the context again to make sure I understand what's relevant.

The context mentions that the paper is a survey on RAG methods, presenting a systematic review of state-of-the-art approaches. It talks about the evolution of RAG through paradigms like naive RAG and effective RAG frameworks. There's also mention of evaluation methods, tasks, datasets, and future directions. The paper is structured with sections on main concepts, paradigms, and research methods, summarizing three main research paradigms from over 100 studies.

So, RAG stands for Retrieval-Augmented Generation. From the context, it seems RAG is a method that combines retrieval of information with generation, likely in the context of large language models (LLMs). The survey discusses different paradigms of RAG, such as naive RAG and more effective frameworks. The key points to include in the answer would be t